In [ ]:
# ============================================
# INSTALL & IMPORT LIBRARIES
# ============================================
!pip install pandas scikit-learn nltk

import pandas as pd
import numpy as np
import re
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ============================================
# LOAD DATASET
# ============================================
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1')

df.columns = df.columns.str.strip().str.lower()

text_col = None
for col in df.columns:
    if 'response' in col or 'feedback' in col or 'text' in col:
        text_col = col
        break

# ============================================
# PREPROCESSING
# ============================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df['cleaned'] = df[text_col].apply(clean_text)

# ============================================
# LABELING
# ============================================
keywords = {
    'Appreciation': ['thank','grateful','appreciate','thankful','good','great','satisfied'],
    'Challenges': ['problem','issue','delay','error','difficult','hard','slow'],
    'Suggestion': ['should','suggest','recommend','improve','better']
}

def assign_labels(text):
    labels = []
    for k, v in keywords.items():
        if any(w in text for w in v):
            labels.append(k)
    return labels if labels else ['Neutral']

df['labels'] = df['cleaned'].apply(assign_labels)

# ============================================
# FEATURES
# ============================================
vectorizer = TfidfVectorizer(ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned'])

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

# ============================================
# MODEL: KNN
# ============================================
model = OneVsRestClassifier(
    KNeighborsClassifier(n_neighbors=5, metric='cosine')
)

model.fit(X, y)

# ============================================
# EVALUATION
# ============================================
y_pred = model.predict(X)
df['predicted_labels'] = mlb.inverse_transform(y_pred)

y_pred_bin = mlb.transform(df['predicted_labels'])

print("Accuracy:", accuracy_score(y, y_pred_bin))
print("F1 Micro:", f1_score(y, y_pred_bin, average='micro'))
print("F1 Macro:", f1_score(y, y_pred_bin, average='macro'))

print(classification_report(y, y_pred_bin, target_names=mlb.classes_))

# ============================================
# PREDICTION
# ============================================
def predict(text, threshold=0.3):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    probs = model.predict_proba(vec)[0]

    labels = [mlb.classes_[i] for i, p in enumerate(probs) if p >= threshold]

    if not labels:
        labels = ["Neutral"]

    return labels, probs

def show_percentages(probs):
    for label, prob in zip(mlb.classes_, probs):
        print(f"{label}: {prob*100:.2f}%")

while True:
    x = input("Enter text (or exit): ")
    if x.lower() == 'exit':
        break

    labels, probs = predict(x)

    print("Predicted Labels:", labels)
    show_percentages(probs)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Saving UAQTEresponses.csv to UAQTEresponses.csv


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.6372549019607843
F1 Micro: 0.6886075949367089
F1 Macro: 0.43552036199095023
              precision    recall  f1-score   support

Appreciation       0.79      0.67      0.72        78
  Challenges       1.00      0.09      0.17        11
     Neutral       0.72      0.77      0.74       107
  Suggestion       1.00      0.06      0.11        17

   micro avg       0.75      0.64      0.69       213
   macro avg       0.88      0.40      0.44       213
weighted avg       0.78      0.64      0.65       213
 samples avg       0.66      0.65      0.65       213

Enter text (or exit): it was great lol
Predicted Labels: ['Appreciation']
Appreciation: 100.00%
Challenges: 20.00%
Neutral: 0.00%
Suggestion: 0.00%
